# WiTA v2 — SigLIP Cross-Modal CTC (Kaggle T4)

First-class V-L recipe per advisor guidance. Frozen SigLIP-So400m vision encoder produces per-frame features; frozen SigLIP text encoder produces per-character prototypes; a small temporal adapter + linear projection is trained against the prototypes via CTC.

| | |
|---|---|
| GPU | NVIDIA T4 (16 GB VRAM), ×2 |
| Encoder | SigLIP-So400m (frozen, ~400M params) |
| Prototypes | SigLIP text encoder over per-char prompts (frozen) |
| Trainable | BiLSTM adapter + Linear projection + log_inv_tau (~5M params) |
| Training cost | features pre-cached once; epochs run on cache → <1 min/epoch |

**Pipeline:** install → setup → stream data → extract SigLIP features ONCE → train head on cache → evaluate

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'transformers>=4.40' --quiet

import sys, os

# Clone the SigLIP branch.  Re-run this cell after a push to refresh.
!rm -rf /kaggle/working/wita_v2
!git clone -b siglip-crossmodal "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'

sys.path.insert(0, '/kaggle/working')

## Cell 2 — Imports & Config

Switch `arch='siglip'` to engage the new path. Defaults are wired so the rest of the notebook just works.

In [ ]:
import os, sys, logging, random
import numpy as np
import torch

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/kaggle/working/logs/run.log'),
    ]
)

from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig
)

FEATURE_CACHE = '/kaggle/working/siglip_features.pt'

cfg = Config(
    data=DataConfig(
        hf_repo_id  = 'yewon816/WiTA',
        lang        = 'english',
        max_zips    = None,        # ← set to 2 for a smoke test
        max_frames  = 64,
    ),
    aug=AugConfig(),               # unused — cached features cannot be augmented
    encoder=EncoderConfig(
        arch                  = 'siglip',
        siglip_model_name     = 'google/siglip-so400m-patch14-384',
        img_size              = 384,           # SigLIP-So400m expects 384×384
        out_dim               = 1152,          # SigLIP-So400m hidden size
        siglip_char_template  = 'the letter {ch}',
        siglip_blank_template = 'no character',
        siglip_sep_template   = 'a brief pause between letters',
        siglip_temporal_arch  = 'lstm',        # 'lstm' | 'conv' | 'transformer' | 'none'
        siglip_adapter_hidden = 512,
        siglip_adapter_layers = 1,
        siglip_init_tau       = 0.07,
        siglip_learnable_tau  = True,
    ),
    recurrent=RecurrentConfig(),   # unused in the SigLIP path; head owns its adapter
    attn=AttnDecoderConfig(),      # unused
    train=TrainConfig(
        num_epochs           = 60,
        batch_size           = 32,         # tiny features, can use bigger batch
        accum_steps          = 1,
        lr                   = 1e-3,       # head only, can be aggressive
        beta2                = 0.999,
        num_workers          = 2,
        optimizer            = 'adamw',
        scheduler            = 'onecycle',
        val_limit            = None,       # cached val is fast; do full pass
        grad_clip            = 5.0,
        unfreeze_after_epoch = 999,        # SigLIP stays frozen always
        backbone_lr_mult     = 1.0,        # only one param group anyway
        checkpoint_dir       = '/kaggle/working/checkpoints',
        resume_path          = None,
        save_frequency       = 10,
        seed                 = 42,
        warmup_pct           = 0.05,
        siglip_feature_cache = FEATURE_CACHE,
    ),
).build()

random.seed(cfg.train.seed)
np.random.seed(cfg.train.seed)
torch.manual_seed(cfg.train.seed)
torch.backends.cudnn.benchmark = True

print(f'Device          : {cfg.device}')
print(f'SigLIP model    : {cfg.encoder.siglip_model_name}')
print(f'Visual dim      : {cfg.encoder.out_dim}')
print(f'Adapter         : {cfg.encoder.siglip_temporal_arch}  hidden={cfg.encoder.siglip_adapter_hidden}')
print(f'CTC vocab       : {cfg.vocab.ctc_vocab_size}  (blank=0, sep={cfg.vocab.sep_idx})')
print(f'Cache path      : {cfg.train.siglip_feature_cache}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)} × {torch.cuda.device_count()}')

## Cell 3 — Stream + index data

Identical to the VideoMAE notebook. Downloads ZIPs one-at-a-time, reads frames into RAM.

In [ ]:
from wita_v2.datasets.dataset import stream_and_index
from wita_v2.datasets.vocab import make_converter

converter = make_converter(cfg.data.lang)
samples   = stream_and_index(cfg)
print(f'Total samples indexed: {len(samples)}')

## Cell 4 — Extract SigLIP features (ONCE)

Runs the frozen SigLIP vision encoder over every frame in the dataset and saves features to disk as fp16. Subsequent training reads from this cache.

Re-runs of the notebook **skip** this cell if the cache already exists. Delete the file to force re-extraction (e.g. when changing SigLIP model or image_size).

In [ ]:
from wita_v2.models.encoders.siglip_encoder import SigLIPVisionEncoder
from wita_v2.datasets.feature_cache import extract_siglip_features

if os.path.exists(FEATURE_CACHE):
    print(f'Cache already exists: {FEATURE_CACHE}  '
          f'({os.path.getsize(FEATURE_CACHE) / 1e6:.1f} MB) — skipping extraction.')
else:
    encoder = SigLIPVisionEncoder(
        model_name = cfg.encoder.siglip_model_name,
        image_size = cfg.encoder.img_size,
    )
    cache = extract_siglip_features(
        samples    = samples,
        encoder    = encoder,
        out_path   = FEATURE_CACHE,
        batch_size = 16,            # frames per SigLIP forward; tune for VRAM
        device     = cfg.device,
        dtype      = torch.float16, # halves disk footprint, negligible loss
    )
    # Drop the encoder + intermediate caches before training.
    del encoder, cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Cell 5 — Build DataLoaders over cached features

In [ ]:
from wita_v2.datasets.feature_cache import make_cached_dataloaders

train_loader, val_loader, cache_meta = make_cached_dataloaders(
    cfg, FEATURE_CACHE, converter=converter,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Source model  : {cache_meta["model_name"]}')
print(f'Feature dim   : {cache_meta["out_dim"]}')

# Sanity-check one batch
feats, labels, input_lens, label_lens = next(iter(train_loader))
print(f'feats         : {feats.shape}  {feats.dtype}')
print(f'labels        : {labels.shape} {labels.dtype}')
print(f'input_lens    : {input_lens.tolist()[:8]}')
print(f'label_lens    : {label_lens.tolist()[:8]}')

gt = converter.decode(
    labels[0, :label_lens[0].item()].int(),
    torch.IntTensor([label_lens[0].item()]),
)
print(f'GT word[0]    : "{gt}"')

## Cell 6 — Build SigLIP cross-modal CTC model

In [ ]:
from wita_v2.models.clip_ctc_model import WiTACLIPCTCModel

# Sync EncoderConfig.out_dim to the dim found in the cache (defensive).
cfg.encoder.out_dim = cache_meta['out_dim']

model = WiTACLIPCTCModel(cfg).to(cfg.device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}  (SigLIP frozen, prototypes frozen)')

# Forward sanity check on the cached batch.
feats_b      = feats.to(cfg.device)
input_lens_b = input_lens.to(cfg.device)
with torch.no_grad():
    ctc_lp, enc_lens = model(feats_b, input_lens_b)
print(f'CTC log_probs   : {ctc_lp.shape}  (B, T, V)')
print(f'enc_lens        : {enc_lens.tolist()[:8]}')
if torch.cuda.is_available():
    print(f'VRAM after fwd  : {torch.cuda.memory_allocated()/1e6:.0f} MB')

## Cell 7 — Train

Uses the existing trainer — model.forward() auto-detects cached features (3D input) and skips the SigLIP encoder. Each epoch processes only the small adapter + projection.

In [ ]:
from wita_v2.training.trainer import train

best_model = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    converter    = converter,
    cfg          = cfg,
)
print('Training complete.')

## Cell 8 — Final evaluation

In [ ]:
from wita_v2.evaluation.evaluator import evaluate_cer, print_sample_table

print('Running full validation (no batch limit) …')
cer, _ = evaluate_cer(
    best_model, val_loader, converter, cfg,
    decode_mode='ctc', max_batches=None,
)
print(f'Final Val CER : {cer:.4f}')

print_sample_table(
    best_model, val_loader, converter, cfg,
    epoch=cfg.train.num_epochs, max_batches=None,
)

print()
print('─' * 50)
print('Reference baselines')
print('  Original WiTA paper (R3D-18 + CTC)        : 0.2924')
print(f'  This run (SigLIP cross-modal + CTC)        : {cer:.4f}')

## Cell 9 — Ablations

Each ablation re-uses the same cached features. Re-run Cells 6→7 after changing config. Total cost per ablation: ~5-10 minutes.

In [ ]:
# ── Adapter ablations ────────────────────────────────────────────────
# cfg.encoder.siglip_temporal_arch = 'transformer'
# cfg.encoder.siglip_temporal_arch = 'conv'
# cfg.encoder.siglip_temporal_arch = 'none'

# ── Prompt-template ablations  (forces feature re-cache if you change
#    these you only need to rebuild PROTOTYPES, not re-encode frames) ──
# cfg.encoder.siglip_char_template = 'a'                             # bare char
# cfg.encoder.siglip_char_template = 'a handwritten letter {ch}'
# cfg.encoder.siglip_char_template = 'the letter {ch} written in the air'

# ── Temperature ablations ────────────────────────────────────────────
# cfg.encoder.siglip_learnable_tau = False
# cfg.encoder.siglip_init_tau      = 0.01      # sharper softmax
# cfg.encoder.siglip_init_tau      = 0.20      # softer

# ── Adapter depth / capacity ─────────────────────────────────────────
# cfg.encoder.siglip_adapter_layers = 2
# cfg.encoder.siglip_adapter_hidden = 1024

print('Edit this cell and re-run Cells 6 → 7 → 8 to run an ablation.')